# Buckeye Corpus: Token-Level Coherence Decay

Replicates the RAID v7 fine-grained token-reveal analysis on **spoken conversational English** from the Buckeye Speech Corpus (26 speakers, Columbus OH).

**Key difference from RAID v7**: Spoken text lacks sentence punctuation, so we use a **fixed token-offset target selection** instead of sentence boundary alignment. This makes the method fully language-agnostic.

**Method**:
1. For each document, sample multiple target regions (30-token spans at 25%, 50%, 75% of document)
2. Reveal context token-by-token backwards from the target (1 to MAX_CONTEXT tokens)
3. Compute perplexity on target for each context length
4. Repeat with token-shuffled context (control for distributional calibration)
5. Corrected marginal = intact marginal - shuffled marginal
6. Fit power law on corrected marginals

**Prediction**: If the 0.75 exponent reflects a universal property of human language production (not just written text), spoken English should show a similar decay rate.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('Imports OK')

In [ ]:
# === Configuration ===
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/buckeye_processed")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/Buckeye_finegrain")
    DRIVE_RAID_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_finegrain")
    # Try Drive first, fall back to upload
    if (DRIVE_DATA / "speaker_concatenated.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/buckeye_processed")
        if not (LOCAL_DATA / "speaker_concatenated.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            print("Upload speaker_concatenated.jsonl:")
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/buckeye_finegrain")
    DATA_DIR = Path("../data/buckeye_processed")
    DRIVE_RAID_RESULTS = Path("../results/raid_finegrain")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True
MAX_CONTEXT = 100       # tokens of context to reveal
TARGET_LEN = 30         # tokens in target region
# Sample 3 target positions per doc (at 25%, 50%, 75% of token stream)
TARGET_FRACTIONS = [0.25, 0.50, 0.75]
MIN_CONTEXT_BEFORE_TARGET = MAX_CONTEXT + 10  # need enough tokens before target
RANDOM_SEED = 42

print(f"Max context: {MAX_CONTEXT} tokens")
print(f"Target length: {TARGET_LEN} tokens")
print(f"Target positions: {TARGET_FRACTIONS}")

In [ ]:
# === Load Buckeye data ===
corpus = []
with open(DATA_DIR / "speaker_concatenated.jsonl") as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"Loaded {len(corpus)} speaker documents")
word_counts = [len(d['text'].split()) for d in corpus]
print(f"Word counts: min={min(word_counts)}, max={max(word_counts)}, mean={np.mean(word_counts):.0f}")
print(f"Total words: {sum(word_counts):,}")

In [ ]:
# === Load model ===
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )
model.eval()
print("Model loaded")

In [ ]:
# === Core functions ===

@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    """Compute perplexity over target region given preceding context."""
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def compute_token_reveal_curve(full_ids, target_start, target_end,
                                shuffled=False, rng_shuf=None):
    """Token-by-token context reveal for a single target region.
    
    No sentence boundary detection needed — context expands backward
    from target_start, one token at a time.
    
    Args:
        full_ids: complete token sequence for the document
        target_start: index of first target token
        target_end: index past last target token
        shuffled: if True, shuffle context tokens (control)
        rng_shuf: random state for shuffling
    """
    target_ids = full_ids[target_start:target_end]
    
    # Context pool = all tokens before the target
    context_pool = list(full_ids[:target_start])
    
    if shuffled and rng_shuf is not None:
        context_pool = list(context_pool)
        rng_shuf.shuffle(context_pool)
    
    max_ctx = min(MAX_CONTEXT, len(context_pool))
    if max_ctx < 10:
        return None
    
    ppls = []
    ctx_lengths = []
    
    for ctx_len in range(1, max_ctx + 1):
        # Take the last ctx_len tokens before target (expanding backward)
        ctx_tokens = context_pool[-ctx_len:]
        
        chunk = ctx_tokens + target_ids
        ppl = compute_ppl(chunk, len(ctx_tokens), len(chunk))
        if not math.isinf(ppl):
            ppls.append(ppl)
            ctx_lengths.append(ctx_len)
    
    if len(ppls) < 10:
        return None
    
    return {
        'ctx_lengths': ctx_lengths,
        'ppls': ppls,
    }


def process_document(doc, rng_shuf):
    """Process one document: sample target positions, compute intact + shuffled curves."""
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    
    intact_curves = []
    shuffled_curves = []
    
    for frac in TARGET_FRACTIONS:
        target_start = int(n * frac)
        target_end = min(target_start + TARGET_LEN, n)
        
        # Need enough context before target
        if target_start < MIN_CONTEXT_BEFORE_TARGET:
            continue
        if target_end - target_start < 5:
            continue
        
        # Intact
        result = compute_token_reveal_curve(full_ids, target_start, target_end,
                                            shuffled=False)
        if result is not None:
            result['doc_id'] = doc['doc_id']
            result['target_frac'] = frac
            intact_curves.append(result)
        
        # Shuffled control
        result_s = compute_token_reveal_curve(full_ids, target_start, target_end,
                                              shuffled=True, rng_shuf=rng_shuf)
        if result_s is not None:
            result_s['doc_id'] = doc['doc_id']
            result_s['target_frac'] = frac
            shuffled_curves.append(result_s)
    
    return intact_curves, shuffled_curves

print("Functions defined")

In [ ]:
# === Run computation (or load cached results) ===
results_path = BASE_DIR / "buckeye_intact_v1.json"
shuffled_path = BASE_DIR / "buckeye_shuffled_v1.json"

if results_path.exists() and shuffled_path.exists():
    with open(results_path) as f:
        all_intact = json.load(f)
    with open(shuffled_path) as f:
        all_shuffled = json.load(f)
    print(f"Loaded {len(all_intact)} intact + {len(all_shuffled)} shuffled curves from cache")
else:
    all_intact = []
    all_shuffled = []
    rng_shuf = np.random.RandomState(RANDOM_SEED + 99)
    
    for doc in tqdm(corpus, desc="Processing speakers"):
        intact, shuffled = process_document(doc, rng_shuf)
        all_intact.extend(intact)
        all_shuffled.extend(shuffled)
    
    # Save results
    with open(results_path, 'w') as f:
        json.dump(all_intact, f)
    with open(shuffled_path, 'w') as f:
        json.dump(all_shuffled, f)
    print(f"Computed {len(all_intact)} intact + {len(all_shuffled)} shuffled curves")

# Summary
unique_docs = set(c['doc_id'] for c in all_intact)
print(f"\nUnique speakers with curves: {len(unique_docs)}")
print(f"Curves per target position:")
for frac in TARGET_FRACTIONS:
    n = sum(1 for c in all_intact if c['target_frac'] == frac)
    print(f"  {frac:.0%}: {n} curves")

## Analysis: Corrected Power Law Fit

Same pipeline as RAID v7:
- Compute raw perplexity curves (intact and shuffled)
- Take marginals (per-token benefit of additional context)
- Subtract shuffled from intact to get **corrected marginals** (pure coherence signal)
- Fit power law on binned corrected marginals

In [ ]:
# === Analysis functions (identical to RAID v7) ===
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def compute_raw_ppl_curve(curves):
    """Mean raw perplexity at each context length (not normalized)."""
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    all_ppl = np.array(all_ppl)
    return np.nanmean(all_ppl, axis=0)

def compute_mean_curve(curves):
    """Normalized mean curve (0-1 scale)."""
    all_norm = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        if ppl[0] - ppl[-1] > 0:
            norm = (ppl[0] - ppl) / (ppl[0] - ppl[-1])
            interp = np.interp(common_x, ctx, norm, left=np.nan, right=np.nan)
            all_norm.append(interp)
    all_norm = np.array(all_norm)
    mean = np.nanmean(all_norm, axis=0)
    sem = np.nanstd(all_norm, axis=0) / np.sqrt(np.sum(~np.isnan(all_norm), axis=0))
    return mean, sem

def fit_power_law(marg):
    """Fit power law to binned marginals. Returns (slope, r, p, bin_centers, bin_means, intercept)."""
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p, bc, bm, intercept
    return None

# Compute curves
intact_ppl = compute_raw_ppl_curve(all_intact)
shuf_ppl = compute_raw_ppl_curve(all_shuffled)

# Marginals
intact_marg = -np.diff(intact_ppl)   # positive = ppl drops = benefit
shuf_marg = -np.diff(shuf_ppl)

# Corrected marginals (pure coherence signal)
corrected_marg = intact_marg - shuf_marg

# Fit
result = fit_power_law(corrected_marg)
if result:
    slope, r, p, bc, bm, intercept = result
    print(f"BUCKEYE SPOKEN (corrected): alpha = {slope:.3f} (r = {r:.3f}, p = {p:.4f})")
else:
    print("WARNING: Power law fit failed")

# Also fit uncorrected for comparison
norm_mean, norm_sem = compute_mean_curve(all_intact)
uncorr_marg = np.diff(norm_mean)
result_uncorr = fit_power_law(uncorr_marg)
if result_uncorr:
    print(f"BUCKEYE SPOKEN (uncorrected): alpha = {result_uncorr[0]:.3f} (r = {result_uncorr[1]:.3f})")

print(f"\nTotal perplexity drop: {intact_ppl[0]:.1f} -> {intact_ppl[-1]:.1f} "
      f"({(1 - intact_ppl[-1]/intact_ppl[0])*100:.1f}% reduction)")

In [ ]:
# === Figure 1: Buckeye standalone results (2x3 panel, same layout as RAID v7) ===
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# A: Raw perplexity curves — intact vs shuffled
ax = axes[0, 0]
ax.plot(common_x, intact_ppl, 'g-', linewidth=2, label='Spoken intact')
ax.plot(common_x, shuf_ppl, 'g:', linewidth=2, label='Spoken shuffled')
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Perplexity')
ax.set_title('A. Raw Perplexity: Intact vs Shuffled', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# B: Marginal gain — intact vs shuffled
ax = axes[0, 1]
ax.plot(common_x[1:], uniform_filter1d(intact_marg, 5), 'g-', linewidth=2, label='Spoken intact')
ax.plot(common_x[1:], uniform_filter1d(shuf_marg, 5), 'g:', linewidth=2, label='Spoken shuffled')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Marginal PPL Drop per Token')
ax.set_title('B. Marginal Gain: Intact vs Shuffled', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# C: Corrected marginal
ax = axes[0, 2]
ax.plot(common_x[1:], uniform_filter1d(corrected_marg, 5), 'g-', linewidth=2, label='Spoken (corrected)')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Corrected Marginal (intact - shuffled)')
ax.set_title('C. Pure Coherence Signal', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# D: Power law fit
ax = axes[1, 0]
result_fit = fit_power_law(corrected_marg)
if result_fit:
    slope, r, p, bc, bm, intercept = result_fit
    ax.plot(bc, bm, 'o-', color='green', linewidth=2, markersize=6, label='Spoken')
    fit_x = np.linspace(min(bc), max(bc), 100)
    fit_y = np.exp(intercept) * fit_x ** slope
    ax.plot(fit_x, fit_y, '--', color='green', alpha=0.5,
            label=f'Spoken: d^{slope:.2f} (r={r:.2f}, p={p:.4f})')
ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens, log)')
ax.set_ylabel('Corrected Marginal Benefit')
ax.set_title('D. Power Law Fit (Corrected)', fontweight='bold')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# E: By target position (stability check)
ax = axes[1, 1]
for frac in TARGET_FRACTIONS:
    frac_curves = [c for c in all_intact if c['target_frac'] == frac]
    frac_shuf = [c for c in all_shuffled if c['target_frac'] == frac]
    if len(frac_curves) < 3:
        continue
    frac_intact_ppl = compute_raw_ppl_curve(frac_curves)
    frac_shuf_ppl = compute_raw_ppl_curve(frac_shuf)
    frac_corr = -np.diff(frac_intact_ppl) - (-np.diff(frac_shuf_ppl))
    frac_fit = fit_power_law(frac_corr)
    if frac_fit:
        s, r_val, p_val, fc_bc, fc_bm, fc_int = frac_fit
        ax.plot(fc_bc, fc_bm, 'o-', markersize=5, label=f'{frac:.0%}: d^{s:.2f} (r={r_val:.2f})')
ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens, log)')
ax.set_ylabel('Corrected Marginal')
ax.set_title('E. Stability: Exponent by Target Position', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# F: Cumulative corrected benefit
ax = axes[1, 2]
cum = np.cumsum(corrected_marg)
if cum[-1] > 0:
    cum_norm = cum / cum[-1]
else:
    cum_norm = cum
ax.plot(common_x[1:], cum_norm, 'g-', linewidth=2, label='Spoken')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Fraction of Total Corrected Benefit')
ax.set_title('F. Cumulative Coherence Benefit', fontweight='bold')
half_idx = np.argmin(np.abs(cum_norm - 0.5)) + 1
ax.axvline(half_idx, color='green', linestyle=':', alpha=0.5)
ax.text(half_idx + 2, 0.45, f'50% at {half_idx} tokens', fontsize=9, color='green')
ax.legend()
ax.grid(True, alpha=0.2)

plt.suptitle('Buckeye Spoken English: Token-Level Coherence Decay',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_buckeye_finegrain.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2: Cross-Modality Comparison (Written vs Spoken)

Load RAID v7 results and overlay power law fits. The key comparison:
- **RAID written human**: alpha = -0.75 (from v7/v4)
- **RAID written AI**: alpha = -1.97
- **Buckeye spoken**: alpha = ???

If spoken English shows alpha near -0.75, the decay exponent is a universal property of human language production, not an artifact of written text structure.

In [ ]:
# === Load RAID v7 results for comparison ===
raid_intact_path = DRIVE_RAID_RESULTS / "finegrain_results_v4.json"
raid_shuffled_path = DRIVE_RAID_RESULTS / "finegrain_shuffled_v4.json"

has_raid = raid_intact_path.exists() and raid_shuffled_path.exists()

if has_raid:
    with open(raid_intact_path) as f:
        raid_all_curves = json.load(f)
    with open(raid_shuffled_path) as f:
        raid_all_shuffled = json.load(f)
    
    raid_human_intact = [c for c in raid_all_curves if c['population'] == 'human']
    raid_ai_intact = [c for c in raid_all_curves if c['population'] == 'ai']
    raid_human_shuffled = [c for c in raid_all_shuffled if c['population'] == 'human']
    raid_ai_shuffled = [c for c in raid_all_shuffled if c['population'] == 'ai']
    
    print(f"RAID results loaded: {len(raid_human_intact)} human, {len(raid_ai_intact)} AI")
    
    # Compute RAID corrected marginals
    rh_intact_ppl = compute_raw_ppl_curve(raid_human_intact)
    rh_shuf_ppl = compute_raw_ppl_curve(raid_human_shuffled)
    ra_intact_ppl = compute_raw_ppl_curve(raid_ai_intact)
    ra_shuf_ppl = compute_raw_ppl_curve(raid_ai_shuffled)
    
    rh_corrected = -np.diff(rh_intact_ppl) - (-np.diff(rh_shuf_ppl))
    ra_corrected = -np.diff(ra_intact_ppl) - (-np.diff(ra_shuf_ppl))
else:
    print("RAID results not found — will plot Buckeye only")
    print(f"  Looked for: {raid_intact_path}")

In [ ]:
# === Figure 2: Cross-modality comparison ===
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Panel A: Power law fits overlaid ---
ax = axes[0]

# Buckeye spoken
bk_fit = fit_power_law(corrected_marg)
if bk_fit:
    s, r, p, bc, bm, inter = bk_fit
    ax.plot(bc, bm, 's-', color='green', linewidth=2, markersize=7,
            label=f'Spoken (Buckeye): {s:.2f}')
    fit_x = np.linspace(min(bc), max(bc), 100)
    ax.plot(fit_x, np.exp(inter) * fit_x**s, '--', color='green', alpha=0.4)

if has_raid:
    # RAID human written
    rh_fit = fit_power_law(rh_corrected)
    if rh_fit:
        s, r, p, bc, bm, inter = rh_fit
        ax.plot(bc, bm, 'o-', color='blue', linewidth=2, markersize=7,
                label=f'Written human (RAID): {s:.2f}')
        ax.plot(fit_x, np.exp(inter) * fit_x**s, '--', color='blue', alpha=0.4)
    
    # RAID AI written
    ra_fit = fit_power_law(ra_corrected)
    if ra_fit:
        s, r, p, bc, bm, inter = ra_fit
        ax.plot(bc, bm, '^-', color='red', linewidth=2, markersize=7,
                label=f'Written AI (RAID): {s:.2f}')
        ax.plot(fit_x, np.exp(inter) * fit_x**s, '--', color='red', alpha=0.4)

ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens)', fontsize=12)
ax.set_ylabel('Corrected Marginal Benefit', fontsize=12)
ax.set_title('A. Power Law Comparison', fontweight='bold', fontsize=13)
ax.legend(fontsize=10, title='Exponent (corrected)')
ax.grid(True, alpha=0.2)

# --- Panel B: Exponent bar chart ---
ax = axes[1]
labels = []
exponents = []
colors = []
rs = []

bk_fit = fit_power_law(corrected_marg)
if bk_fit:
    labels.append('Spoken\n(Buckeye)')
    exponents.append(bk_fit[0])
    colors.append('green')
    rs.append(bk_fit[1])

if has_raid:
    rh_fit = fit_power_law(rh_corrected)
    if rh_fit:
        labels.append('Written human\n(RAID)')
        exponents.append(rh_fit[0])
        colors.append('blue')
        rs.append(rh_fit[1])
    
    ra_fit = fit_power_law(ra_corrected)
    if ra_fit:
        labels.append('Written AI\n(RAID)')
        exponents.append(ra_fit[0])
        colors.append('red')
        rs.append(ra_fit[1])

# Add Anderson & Schooler reference
labels.append('Anderson &\nSchooler (1991)')
exponents.append(-0.77)
colors.append('gray')
rs.append(None)

bars = ax.bar(range(len(labels)), exponents, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Power Law Exponent', fontsize=12)
ax.set_title('B. Decay Exponents Across Modalities', fontweight='bold', fontsize=13)
ax.axhline(-0.77, color='gray', linestyle=':', alpha=0.5, label='Anderson & Schooler -0.77')
ax.grid(True, alpha=0.2, axis='y')

for i, (exp, r_val) in enumerate(zip(exponents, rs)):
    label = f'{exp:.2f}'
    if r_val is not None:
        label += f'\n(r={r_val:.2f})'
    ax.text(i, exp - 0.08, label, ha='center', fontsize=9, fontweight='bold')

# --- Panel C: Corrected marginals smoothed overlay ---
ax = axes[2]
ax.plot(common_x[1:], uniform_filter1d(corrected_marg, 5), 'g-', linewidth=2,
        label='Spoken (Buckeye)')
if has_raid:
    ax.plot(common_x[1:], uniform_filter1d(rh_corrected, 5), 'b-', linewidth=2,
            label='Written human (RAID)')
    ax.plot(common_x[1:], uniform_filter1d(ra_corrected, 5), 'r--', linewidth=2,
            label='Written AI (RAID)')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('Corrected Marginal', fontsize=12)
ax.set_title('C. Coherence Signal: Written vs Spoken', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

plt.suptitle('Cross-Modality Coherence Decay: Written vs Spoken English',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig2_cross_modality.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary table
print("\n" + "="*60)
print("CROSS-MODALITY COMPARISON")
print("="*60)
for label, exp in zip(labels, exponents):
    print(f"  {label.replace(chr(10), ' '):<30}: alpha = {exp:.3f}")

## Per-Speaker Analysis

Check whether the exponent is stable across individual speakers, or if there's meaningful variance (which could correlate with speaker characteristics).

In [ ]:
# === Per-speaker exponent analysis ===
speaker_ids = sorted(set(c['doc_id'] for c in all_intact))
speaker_exponents = []

for sid in speaker_ids:
    sp_intact = [c for c in all_intact if c['doc_id'] == sid]
    sp_shuf = [c for c in all_shuffled if c['doc_id'] == sid]
    
    if len(sp_intact) < 2 or len(sp_shuf) < 2:
        continue
    
    sp_intact_ppl = compute_raw_ppl_curve(sp_intact)
    sp_shuf_ppl = compute_raw_ppl_curve(sp_shuf)
    sp_corr = -np.diff(sp_intact_ppl) - (-np.diff(sp_shuf_ppl))
    
    sp_fit = fit_power_law(sp_corr)
    if sp_fit:
        speaker_exponents.append({
            'speaker_id': sid,
            'exponent': sp_fit[0],
            'r': sp_fit[1],
            'p': sp_fit[2],
            'n_curves': len(sp_intact),
        })

if speaker_exponents:
    df_sp = pd.DataFrame(speaker_exponents)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram of exponents
    ax = axes[0]
    ax.hist(df_sp['exponent'], bins=15, color='green', alpha=0.7, edgecolor='black')
    ax.axvline(df_sp['exponent'].mean(), color='darkgreen', linestyle='-', linewidth=2,
               label=f'Mean: {df_sp["exponent"].mean():.2f}')
    ax.axvline(df_sp['exponent'].median(), color='darkgreen', linestyle='--', linewidth=2,
               label=f'Median: {df_sp["exponent"].median():.2f}')
    ax.axvline(-0.75, color='blue', linestyle=':', linewidth=2,
               label='RAID written: -0.75')
    ax.axvline(-0.77, color='gray', linestyle=':', linewidth=2,
               label='Anderson & Schooler: -0.77')
    ax.set_xlabel('Decay Exponent', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Distribution of Per-Speaker Exponents', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
    
    # Sorted exponent plot with CIs
    ax = axes[1]
    df_sorted = df_sp.sort_values('exponent').reset_index(drop=True)
    ax.bar(range(len(df_sorted)), df_sorted['exponent'], color='green', alpha=0.6)
    ax.axhline(df_sp['exponent'].mean(), color='darkgreen', linestyle='-', linewidth=2)
    ax.axhline(-0.75, color='blue', linestyle=':', linewidth=2, label='RAID written human')
    ax.axhline(-1.97, color='red', linestyle=':', linewidth=2, label='RAID written AI')
    ax.set_xlabel('Speaker (sorted)', fontsize=12)
    ax.set_ylabel('Decay Exponent', fontsize=12)
    ax.set_title('Per-Speaker Exponents (sorted)', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'fig3_per_speaker.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nPer-speaker exponents (n={len(df_sp)}):")
    print(f"  Mean:   {df_sp['exponent'].mean():.3f} +/- {df_sp['exponent'].std():.3f}")
    print(f"  Median: {df_sp['exponent'].median():.3f}")
    print(f"  Range:  [{df_sp['exponent'].min():.3f}, {df_sp['exponent'].max():.3f}]")
    print(f"  Significant fits (p<0.05): {(df_sp['p'] < 0.05).sum()} / {len(df_sp)}")
else:
    print("Not enough per-speaker data for individual fits")

## Summary

Key numbers to report:
- Buckeye corrected exponent: see Panel D above
- Comparison to RAID written human (-0.75) and AI (-1.97)
- Anderson & Schooler (1991) reference: -0.77
- Per-speaker stability: mean, SD, range
- If spoken exponent is near -0.75, this supports the claim that coherence decay is a universal property of human language production constrained by working memory, not an artifact of written text conventions.